In [1]:
#Upload the raw dataset into Google Colab
from google.colab import files

#Import pandas for data cleaning and manipulation
import pandas as pd

uploaded = files.upload()

#Load the raw dataset into a DataFrame
df = pd.read_csv("frailty_data_clean.csv")

#Preview the first five rows
df.head()

Saving frailty_data_clean.csv to frailty_data_clean.csv


,Height,Weight,Age,Grip strength,Frailty
0,65.8,112,30,30,N
1,71.5,136,19,31,N
2,69.4,153,45,29,N
3,68.2,142,22,28,Y
4,67.8,144,29,24,Y


In [2]:
#Convert height from inches to meters
df["Height_m"] = df["Height"] * 0.0254

#Confirm the Height_m column was added to the DataFrame
df.head()

,Height,Weight,Age,Grip strength,Frailty,Height_m
0,65.8,112,30,30,N,1.67132
1,71.5,136,19,31,N,1.81610
2,69.4,153,45,29,N,1.76276
3,68.2,142,22,28,Y,1.73228
4,67.8,144,29,24,Y,1.72212


In [3]:
#Convert weight from pounds to kilograms
df["Weight_kg"] = df["Weight"] * 00.45359237

#Confirm the Weight_kg column was added to the DataFrame
df.head()

,Height,Weight,Age,Grip strength,Frailty,Height_m,Weight_kg
0,65.8,112,30,30,N,1.67132,50.802345
1,71.5,136,19,31,N,1.81610,61.688562
2,69.4,153,45,29,N,1.76276,69.399633
3,68.2,142,22,28,Y,1.73228,64.410117
4,67.8,144,29,24,Y,1.72212,65.317301


In [4]:
#Calculate BMI
df["BMI"] = (df["Weight_kg"] / (df["Height_m"] ** 2)).round(2)

#Confirm the BMI column was added to the DataFrame
df.head()

,Height,Weight,Age,Grip strength,Frailty,Height_m,Weight_kg,BMI
0,65.8,112,30,30,N,1.67132,50.802345,18.19
1,71.5,136,19,31,N,1.81610,61.688562,18.70
2,69.4,153,45,29,N,1.76276,69.399633,22.33
3,68.2,142,22,28,Y,1.73228,64.410117,21.46
4,67.8,144,29,24,Y,1.72212,65.317301,22.02


In [5]:
#Create categories for age groups
df["AgeGroup"] = pd.cut(
df["Age"],
bins=[0, 29, 45, 60, float("inf")],
labels=["<30", "30-45", "46-60", ">60"]
)

#Confirm AgeGroup categories were added correctly
df[["Age", "AgeGroup"]]

,Age,AgeGroup
0,30,30-45
1,19,<30
2,45,30-45
3,22,<30
4,29,<30
5,50,46-60
6,51,46-60
7,23,<30
8,17,<30
9,39,30-45


In [6]:
#Convert Frailty from Y/N to binary values 1/0 and store as int82
df["Frailty_binary"] = df["Frailty"].map({"Y": 1, "N": 0}).astype("int8")

#Confirm Frailty binary encoding
df[["Frailty", "Frailty_binary"]]

#Confirm Frailty_binary data type
df["Frailty_binary"].dtype

dtype('int8')

In [7]:
#Confirm the Frailty_binary column was added to the DataFrame
df.head()

,Height,Weight,Age,Grip strength,Frailty,Height_m,Weight_kg,BMI,AgeGroup,Frailty_binary
0,65.8,112,30,30,N,1.67132,50.802345,18.19,30-45,0
1,71.5,136,19,31,N,1.81610,61.688562,18.70,<30,0
2,69.4,153,45,29,N,1.76276,69.399633,22.33,30-45,0
3,68.2,142,22,28,Y,1.73228,64.410117,21.46,<30,1
4,67.8,144,29,24,Y,1.72212,65.317301,22.02,<30,1


In [8]:
#Create dummy variables for each AgeGroup category
age_dummies = pd.get_dummies(
df["AgeGroup"],
prefix="AgeGroup",
dtype="int8"
)

#Add AgeGroup dummy variable columns to the DataFrame
df = pd.concat([df, age_dummies], axis=1)

#Confirm AgeGroup dummy variables were added to the DataFrame
df.head()

,Height,Weight,Age,Grip strength,Frailty,Height_m,Weight_kg,BMI,AgeGroup,Frailty_binary,AgeGroup_<30,AgeGroup_30-45,AgeGroup_46-60,AgeGroup_>60
0,65.8,112,30,30,N,1.67132,50.802345,18.19,30-45,0,0,1,0,0
1,71.5,136,19,31,N,1.81610,61.688562,18.70,<30,0,1,0,0,0
2,69.4,153,45,29,N,1.76276,69.399633,22.33,30-45,0,0,1,0,0
3,68.2,142,22,28,Y,1.73228,64.410117,21.46,<30,1,1,0,0,0
4,67.8,144,29,24,Y,1.72212,65.317301,22.02,<30,1,1,0,0,0


In [9]:
#Create a summary table with the mean, median, and standard deviation
numeric_columns = df.select_dtypes(include="number").columns
summary_table = df[numeric_columns].agg(["mean", "median", "std"]).T

#Round the summary statistics to two decimal places
summary_table = summary_table.round(2)

#Display the summary table
summary_table

,mean,median,std
Height,68.60,68.45,1.67
Weight,131.90,136.00,14.23
Age,32.50,29.50,12.86
Grip strength,26.00,27.00,4.52
Height_m,1.74,1.74,0.04
Weight_kg,59.83,61.69,6.46
BMI,19.68,19.19,1.78
Frailty_binary,0.40,0.00,0.52
AgeGroup_<30,0.50,0.50,0.53
AgeGroup_30-45,0.30,0.00,0.48


In [10]:
#Calculate correlation between grip strength and frailty
correlation = df["Grip strength"].corr(df["Frailty_binary"])

#Display correlation rounded to three decimal places
print("Correlation:", round(correlation, 3))

Correlation: -0.476


In [11]:
#Create the results folder
import os
os.makedirs("results", exist_ok=True)

#Confirm the results folder exists
print(os.path.exists("results"))

True


In [12]:
#Save summary statistics and correlation results to findings.md
with open("results/findings.md", "w") as file:
  file.write("# Assignment 1 Findings\n\n")
  file.write("## Question 1, Part D(I): Summary Statistics\n\n")
  file.write(summary_table.to_markdown())
  file.write("\n\n")
  file.write("## Question 1, Part D(II): Grip Strength and Frailty Correlation\n\n")
  file.write(
      f"Correlation between Grip Strength and Frailty_binary: "
      f"**{correlation:.3f}**\n"
      )

In [22]:
#Confirm that the findings file was created
print(os.path.exists("results/findings.md"))

True


In [23]:
#Preview the findings file
with open("results/findings.md", "r") as file:
  print(file.read())

# Assignment 1 Findings

## Question 1, Part D(I): Summary Statistics

|                |   mean |   median |   std |
|:---------------|-------:|---------:|------:|
| Height         |  68.6  |    68.45 |  1.67 |
| Weight         | 131.9  |   136    | 14.23 |
| Age            |  32.5  |    29.5  | 12.86 |
| Grip strength  |  26    |    27    |  4.52 |
| Height_m       |   1.74 |     1.74 |  0.04 |
| Weight_kg      |  59.83 |    61.69 |  6.46 |
| BMI            |  19.68 |    19.19 |  1.78 |
| Frailty_binary |   0.4  |     0    |  0.52 |
| AgeGroup_<30   |   0.5  |     0.5  |  0.53 |
| AgeGroup_30-45 |   0.3  |     0    |  0.48 |
| AgeGroup_46-60 |   0.2  |     0    |  0.42 |
| AgeGroup_>60   |   0    |     0    |  0    |

## Question 1, Part D(II): Grip Strength and Frailty Correlation

Correlation between Grip Strength and Frailty_binary: **-0.476**



In [25]:
#Download the findings file to your computer
from google.colab import files
files.download("results/findings.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
#Save the final analyzed dataset as a CSV file
df.to_csv("frailty_data_analyzed.csv", index=False)

In [28]:
#Download the final analyzed dataset
files.download("frailty_data_analyzed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>